In [8]:
from tennis_data_pipeline.datasources.wta import WtaApiClient

# WtaApiClient wraps the ad hoc requests calls from earlier exploration:
# no custom headers needed, handles pagination, and returns a flat DataFrame
# keyed on `tournament_group_id` (confirmed stable across years).
YEAR = 2024
client = WtaApiClient()
df_tournaments = client.get_tournaments(YEAR)
print(list(df_tournaments.columns))
df_tournaments

['tournament_group_id', 'group_name', 'level', 'title', 'year', 'start_date', 'end_date', 'surface', 'in_outdoor', 'city', 'country', 'singles_draw_size', 'doubles_draw_size', 'prize_money', 'singles_champion']


,tournament_group_id,group_name,level,title,year,start_date,end_date,surface,in_outdoor,city,country,singles_draw_size,doubles_draw_size,prize_money,singles_champion
0,2084,UNITED CUP,WTA 500,"United Cup - Australia, AUS",2024,2023-12-29,2024-01-07,Hard,O,SYDNEY + PERTH,AUSTRALIA,0,0,5000000,NaN
1,800,BRISBANE,WTA 500,Brisbane International presented by Evie - Bri...,2024,2023-12-31,2024-01-07,Hard,O,BRISBANE,AUSTRALIA,48,24,1736763,Elena Rybakina
2,4381,ARCADIA,ITF,"ITF/USTA W35 - ARCADIA, CA, USA",2024,2024-01-01,2024-01-07,Hard,O,,UNITED STATES,32,16,25000,Fiona Crawley
3,2742,MONASTIR,ITF,"ITF W15- MONASTIR, TUNISIA",2024,2024-01-01,2024-01-07,Hard,O,,TUNISIA,32,16,15000,Amelie Smejkalova
4,2096,CANBERRA 125,WTA 125,"Workday Canberra International - Canberra, AUS",2024,2024-01-01,2024-01-06,Hard,O,CANBERRA,AUSTRALIA,32,16,164000,Nuria Parrizas Diaz
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710,800,BRISBANE,WTA 500,Brisbane International presented by Evie - Bri...,2025,2024-12-29,2025-01-05,Hard,O,BRISBANE,AUS,48,16,1520600,Aryna Sabalenka
711,698,NONTHABURI,ITF,"WTT W75 - NONTHABURI, THAILAND",2025,2024-12-30,2025-01-05,Hard,O,,THA,32,16,60000,Kyoka Okamura
712,1049,AUCKLAND,WTA 250,"ASB Classic - Auckland, NZL",2025,2024-12-30,2025-01-05,Hard,O,AUCKLAND,NZL,32,16,275094,Clara Tauson
713,2096,CANBERRA 125,WTA 125,"Workday Canberra International - Canberra, AUS",2025,2024-12-30,2025-01-04,Hard,O,CANBERRA,AUS,32,16,200000,Aoi Ito


In [7]:
print(f"{len(df_tournaments)} tournaments returned for {YEAR}")

715 tournaments returned for 2024


In [5]:
print(df_tournaments["level"].value_counts())

# Tennis-Data UK only covers the main tour (not ITF/W15-W100 events), so this is
# likely the subset we'd actually try to match against.
main_tour = df_tournaments[~df_tournaments["level"].eq("ITF")]
main_tour[[
    "tournament_group_id", 
    "group_name", 
    "level", 
    "title", 
    "city", 
    "start_date", 
    "end_date"
]]

level
ITF           647
WTA 125        49
WTA 250        19
WTA 500        17
WTA 1000       10
Grand Slam      4
Finals          1
Name: count, dtype: int64


,tournament_group_id,group_name,level,title,city,start_date,end_date
0,2084,UNITED CUP,WTA 500,"United Cup - Australia, AUS",SYDNEY + PERTH,2024-12-27,2025-01-05
1,800,BRISBANE,WTA 500,Brisbane International presented by Evie - Bri...,BRISBANE,2024-12-29,2025-01-05
2,2096,CANBERRA 125,WTA 125,"Workday Canberra International - Canberra, AUS",CANBERRA,2024-12-30,2025-01-04
3,1049,AUCKLAND,WTA 250,"ASB Classic - Auckland, NZL",AUCKLAND,2024-12-30,2025-01-05
6,2014,ADELAIDE,WTA 500,"Adelaide International - Adelaide, AUS",ADELAIDE,2025-01-06,2025-01-11
...,...,...,...,...,...,...,...
694,2076,COLINA 125,WTA 125,"LP Open by IND - Colina, CHI",COLINA,2025-11-17,2025-11-23
709,2052,BUENOS AIRES 125,WTA 125,"IEB+ Argentina Open - Buenos Aires, ARG",BUENOS AIRES,2025-11-24,2025-11-30
722,1118,QUITO 125,WTA 125,"Quito Open - Quito, ECU",QUITO,2025-12-01,2025-12-07
723,2056,ANGERS 125,WTA 125,"Open Angers Loire Trélazé - Angers, FRA",ANGERS,2025-12-01,2025-12-07


In [8]:
# Check whether `tournamentGroup.id` is stable year-over-year for the same
# tournament (this is the key question - if stable, it's a much better anchor
# id than fuzzy name-matching).
df_2019 = client.get_tournaments(2019)

check_names = ["ADELAIDE", "INDIAN WELLS", "DUBAI", "DOHA", "AUCKLAND"]
compare = (
    df_tournaments[df_tournaments["group_name"].isin(check_names)][["group_name", "tournament_group_id"]]
    .drop_duplicates()
    .merge(
        df_2019[df_2019["group_name"].isin(check_names)][
            ["group_name", "tournament_group_id"]
        ].drop_duplicates(),
        on="group_name",
        suffixes=("_2025", "_2019"),
        how="outer",
    )
)
compare

,group_name,tournament_group_id_2025,tournament_group_id_2019
0,ADELAIDE,2014,NaN
1,AUCKLAND,1049,1049.0
2,DOHA,1003,1003.0
3,DUBAI,718,718.0
4,DUBAI,718,2300.0
5,DUBAI,2300,718.0
6,DUBAI,2300,2300.0
7,INDIAN WELLS,609,609.0


## Building the full 2026 tournament table

`tournament_group_id` is confirmed stable across years (above), so it can serve
as `official_tournament_id` directly - no fuzzy name-matching needed, unlike
the Sackmann/UK crosswalk (see `mapper.tournaments`).

The API caps `pageSize` at 100 server-side regardless of what's requested
(confirmed empirically - the response always comes back with `pageInfo.pageSize
== 100` even when we ask for 1000, and `numPages` is unreliably `0`), so paging
has to continue until a page comes back empty, not until `numPages` is
reached. `WtaApiClient.get_tournaments()` already handles this.

This time we want *every* level (ITF included) - filtering to main-tour-only
is a downstream concern, not something to bake into the raw pull.


In [1]:
from tennis_data_pipeline.workflows.wta_api import build_wta_api_tournaments

# Same client -> workflow -> script pattern as the UK/Sackmann sources:
# handler.wta_api.tournaments.build_wta_api_tournament_table does the
# renaming/typing (tournament_group_id -> official_tournament_id), and this
# workflow upserts the result into the shared clean CSV.
table_path, tournament_count = build_wta_api_tournaments([2026])
print(f"{tournament_count} tournament(s) now in {table_path}")


542 tournament(s) now in /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline/data/clean/wta_api/tournaments/wta_api_tournaments.csv


In [2]:
import pandas as pd

df_2026 = pd.read_csv(table_path)
print(df_2026.shape)
print(df_2026["level"].value_counts())
df_2026.sort_values("start_date").head(20)


(542, 16)
level
ITF           420
WTA 125        67
WTA 250        22
WTA 500        18
WTA 1000       10
Grand Slam      4
Finals          1
Name: count, dtype: int64


,tour,year,official_tournament_id,group_name,level,title,surface,in_outdoor,city,country,singles_draw_size,doubles_draw_size,prize_money,start_date,end_date,singles_champion
346,wta,2026,4440,NAIROBI,ITF,"WTT W35 - NAIROBI, KENYA",Clay,O,NaN,KEN,32,16,30000,2025-12-29,2026-01-04,Angella Okutoyi
495,wta,2026,4948,AHMEDABAD,ITF,"WTT W15 - AHMEDABAD, INDIA",Hard,O,NaN,IND,32,16,15000,2025-12-29,2026-01-04,Mariia Golovina
251,wta,2026,2742,MONASTIR,ITF,"WTT W15 - MONASTIR, TUNISIA",Hard,O,NaN,TUN,32,16,15000,2025-12-29,2026-01-04,Lan Mi
196,wta,2026,2084,UNITED CUP,WTA 500,"United Cup - Australia, AUS",Hard,O,PERTH + SYDNEY,AUS,0,0,5903345,2026-01-02,2026-01-11,NaN
71,wta,2026,800,BRISBANE,WTA 500,"Brisbane International - Brisbane, AUS",Hard,O,BRISBANE,AUS,48,16,1691602,2026-01-04,2026-01-11,Aryna Sabalenka
91,wta,2026,911,ANTALYA,ITF,"WTT W35 - ANTALYA, TURKEY",Clay,O,NaN,TUR,32,16,30000,2026-01-05,2026-01-11,Andreea Prisacariu
65,wta,2026,734,NIAROBI 2,ITF,"WTT W35 - NAIROBI 2, KENYA",Clay,O,NaN,KEN,32,16,30000,2026-01-05,2026-01-11,Angella Okutoyi
108,wta,2026,1049,AUCKLAND,WTA 250,"ASB Classic - Auckland, NZL",Hard,O,AUCKLAND,NZL,32,16,283347,2026-01-05,2026-01-11,Elina Svitolina
486,wta,2026,4902,HURGHADA,ITF,"WTT W15 - HURGHADA, EGYPT",Hard,O,NaN,EGY,32,16,15000,2026-01-05,2026-01-11,Daria Egorova
272,wta,2026,2927,MONASTIR 2,ITF,"WTT W15 - MONASTIR 2, TUNISIA",Hard,O,NaN,TUN,32,16,15000,2026-01-05,2026-01-11,Carolyn Ansari
